In [2]:
import pandas as pd 
import json

In [3]:
#Need to load in JSON dictionaries in order to work with pandas
#starting with one slice for testing 
with open("../data/mpd.slice.0-999.json") as f:
    data = json.load(f)

In [ ]:
#This pandas method transforms the nested dictionaries into a flat table
pd.json_normalize(data)

,playlists,info.generated_on,info.slice,info.version
0,"[{'name': 'Throwbacks', 'collaborative': 'fals...",2017-12-03 08:41:42.057563,0-999,v1


In [4]:
#For the bronze layer we want to have the important data in its raw form
#In this circumstance we care about the playlists, so we use the record_path parameter to specify 
normaldata = pd.json_normalize(data, record_path="playlists") #json_normalize use allows us to unwrap nested data according to where we have our records using the 'record_path=' parameter
#which in this case is the playlists
print(type(normaldata))
normaldata
#Note that this is missing the metadata for the slice

<class 'pandas.DataFrame'>


,name,collaborative,pid,modified_at,num_tracks,num_albums,num_followers,tracks,num_edits,duration_ms,num_artists,description
0,Throwbacks,false,0,1493424000,52,47,1,"[{'pos': 0, 'artist_name': 'Missy Elliott', 't...",6,11532414,37,NaN
1,Awesome Playlist,false,1,1506556800,39,23,1,"[{'pos': 0, 'artist_name': 'Survivor', 'track_...",5,11656470,21,NaN
2,korean,false,2,1505692800,64,51,1,"[{'pos': 0, 'artist_name': 'Hoody', 'track_uri...",18,14039958,31,NaN
3,mat,false,3,1501027200,126,107,1,"[{'pos': 0, 'artist_name': 'Camille Saint-Saën...",4,28926058,86,NaN
4,90s,false,4,1401667200,17,16,2,"[{'pos': 0, 'artist_name': 'The Smashing Pumpk...",7,4335282,16,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
995,old,false,995,1507852800,41,40,1,"[{'pos': 0, 'artist_name': 'Katrina', 'track_u...",8,9917901,36,NaN
996,Daze,false,996,1479254400,17,17,1,"[{'pos': 0, 'artist_name': 'PARTYNEXTDOOR', 't...",13,3699248,15,NaN
997,rap,false,997,1410307200,119,98,1,"[{'pos': 0, 'artist_name': 'LoveRance', 'track...",63,27538723,82,NaN
998,Country,false,998,1507939200,108,75,1,"[{'pos': 0, 'artist_name': 'Hunter Hayes', 'tr...",37,24950143,40,NaN


## To follow the best practices of a standard medallion architecture- we will also keep the meta data for slices in an 'info' table and a separate table for playlists

In [ ]:
from pathlib import Path

datapath = Path('../data/') #Path containing all slices

#The following took 17 minutes to finish - indicating that invoking to_csv on EVERY loop interation the OS has to write (append) to SSD/HDD which is the biggest bottleneck
header = True #Ensuring header is written only on the first iteration
with open ("../bronze/sliceinfo.csv", "a", newline="") as file:
    for i in datapath.iterdir():
        with open(i,'r') as f:
            jread = json.load(f)
        df = pd.json_normalize(jread)
        df.to_csv(file, header=header)
        header = False

In [ ]:
# Attempt will be to append to all data to a list (using RAM here), convert it to a datafram and perform one write operation 
#Per pandas- do not use CONCAT repeatedly as every call makes a copy of the data

# Warning ! This crashed my Lenovo - not enough RAM
from pathlib import Path

spdatapath = Path("../data/")

df = pd.DataFrame() #instantiate dempty dataframe 
df_list = []

for slice in spdatapath.iterdir(): #iterate through data directory (path object - posix in Lenovo)
    with open(slice,'r') as f: #opens slice on current iteration
        sliceread= json.load(f) #reads nested JSON
        df = pd.json_normalize(sliceread) #standardizes and convert JSON data of current slice to dataframe
        df_list.append(df) #appending 



In [ ]:
# May have reached bottle neck of working with csv files- computation is too expensive 

# Will use parquet file processing to compare (using old csv logic) - PyArrow will be the engine used by pandas
from pathlib import Path

datapath = Path('../data/') #Path containing all slices

for i in datapath.iterdir(): #iterate through directory using path .iterdir() method
    with open(i,'r') as f: #open file on current iteration (going from path object to json file)
        jread = json.load(f) #loads the nested json
    df = pd.json_normalize(jread) #normalizes and flattens the json 
    df.to_parquet(f'../bronze/{i.stem}.parquet') #converts to parquet file type and stores it in bronze/

# The above took 7 minutes to process- More than 50% computation time spent vs csv

In [4]:
#checking if one of the slices is folllowing schema, next step will be to use pyspark to read file 
pd.read_parquet('../bronze/mpd.slice.0-999.parquet')

,playlists,info.generated_on,info.slice,info.version
0,"[{'collaborative': 'false', 'description': Non...",2017-12-03 08:41:42.057563,0-999,v1


In [5]:
pd.read_parquet('../bronze/mpd.slice.1000-1999.parquet')

,playlists,info.generated_on,info.slice,info.version
0,"[{'collaborative': 'false', 'description': Non...",2017-12-03 08:41:42.057563,1000-1999,v1


In [3]:
#Pyspark Implementation (In-Progress)
#Need Java installed
#Pyspark Installation -- pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadingTest").getOrCreate() #Standard way of starting a pyspark application 


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 13:13:17 WARN Utils: Your hostname, g0dzi114, resolves to a loopback address: 127.0.1.1; using 192.168.1.16 instead (on interface wlp3s0)
26/03/28 13:13:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 13:13:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.parquet("../bronze/") #reads in the entire bronze directory ->Notice this takes 3 seconds

In [ ]:
df.printSchema() #This is the current schema automatically set by spark when json was converted to parquet

root
 |-- playlists: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- collaborative: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- duration_ms: long (nullable = true)
 |    |    |-- modified_at: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- num_albums: long (nullable = true)
 |    |    |-- num_artists: long (nullable = true)
 |    |    |-- num_edits: long (nullable = true)
 |    |    |-- num_followers: long (nullable = true)
 |    |    |-- num_tracks: long (nullable = true)
 |    |    |-- pid: long (nullable = true)
 |    |    |-- tracks: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- album_name: string (nullable = true)
 |    |    |    |    |-- album_uri: string (nullable = true)
 |    |    |    |    |-- artist_name: string (nullable = true)
 |    |    |    |    |-- artist_uri: string (nullable = true)
 |

In [ ]:
df.show(10) #data is seemingly not in any particular order...info.slice shows various slices appearing in the first 10 rows... operation also took about 29 seconds

+--------------------+--------------------+-------------+------------+
|           playlists|   info.generated_on|   info.slice|info.version|
+--------------------+--------------------+-------------+------------+
|[{false, NULL, 13...|2017-12-03 08:41:...|609000-609999|          v1|
|[{false, NULL, 13...|2017-12-03 08:41:...|604000-604999|          v1|
|[{false, NULL, 17...|2017-12-03 08:41:...|348000-348999|          v1|
|[{false, NULL, 20...|2017-12-03 08:41:...|  45000-45999|          v1|
|[{true, NULL, 809...|2017-12-03 08:41:...|243000-243999|          v1|
|[{false, NULL, 22...|2017-12-03 08:41:...|268000-268999|          v1|
|[{false, NULL, 39...|2017-12-04 03:05:...|850000-850999|          v1|
|[{false, NULL, 61...|2017-12-03 08:41:...|335000-335999|          v1|
|[{false, NULL, 18...|2017-12-04 03:05:...|759000-759999|          v1|
|[{false, NULL, 10...|2017-12-03 08:41:...|300000-300999|          v1|
+--------------------+--------------------+-------------+------------+
only s

In [ ]:
#goal now is to verify that data in parquet files is as expected before proceeding to silver table implementation
df.count() #->Returns 1000 rows as expected 

1000

In [ ]:
#Bronze schema should ideally be two tables -> Playlists and Info
#Current schema is just one tree structure -> To follow two-table framework, schema could be split by making two directories inside bronze (playlists/, info/)
#Need to rewrite parquet files into separate folders with intended level at conversion
#Observing JSON data- The playlists are stored in the playlist array, with each index representing a playlist object.
#Each playlist object has 8 name value pairs, with the 8th being a tracks array object with every index being a track object containing track data
df.printSchema()

root
 |-- playlists: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- collaborative: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- duration_ms: long (nullable = true)
 |    |    |-- modified_at: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- num_albums: long (nullable = true)
 |    |    |-- num_artists: long (nullable = true)
 |    |    |-- num_edits: long (nullable = true)
 |    |    |-- num_followers: long (nullable = true)
 |    |    |-- num_tracks: long (nullable = true)
 |    |    |-- pid: long (nullable = true)
 |    |    |-- tracks: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- album_name: string (nullable = true)
 |    |    |    |    |-- album_uri: string (nullable = true)
 |    |    |    |    |-- artist_name: string (nullable = true)
 |    |    |    |    |-- artist_uri: string (nullable = true)
 |